## Validate data from API using Pydantic

a) Create a Pydantic model with name Joke with the following fields
- id with type integer
- joke with type string

b) Validate the data from the API using the Joke model. Test out your Joke instance to see that you can access the joke and id fields.

c) Now create a new Joke Pydantic model that also have the field words_in_joke. This is a computed field and a property

Note that computed_field is imported from pydantic. Validate a random joke with your new Joke model.

d) Request 10 jokes from the api and validate them into many Jokes instances that you store into a list. Make sure to use sleep for 5 seconds to not request from the API too much.

In [ ]:
import requests
import time
from pydantic import BaseModel, ValidationError, computed_field

headers = {"Accept": "application/json"}
response = requests.get("https://icanhazdadjoke.com", headers=headers)
data = response.json()

print(response.json())

{'id': 'A5189xcFIBd', 'joke': '"Dad, do you think it\'s going to snow this winter?" "I dont know, its all up in the air"', 'status': 200}


In [53]:
data.keys()

dict_keys(['id', 'joke', 'status'])

In [22]:
class Joke(BaseModel):
    id: str
    joke: str

try:
    Joke(id = 444354, joke = 543313)
except ValidationError as err:
    print(err)

2 validation errors for Joke
id
  Input should be a valid string [type=string_type, input_value=444354, input_type=int]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type
joke
  Input should be a valid string [type=string_type, input_value=543313, input_type=int]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type


In [27]:
class Joke(BaseModel):
    id: str
    joke: str

    @computed_field
    def words_in_joke(self) -> int:
        return len(self.joke.split())

joke_length = Joke.model_validate(data)
joke_length

Joke(id='daaUfibh', joke='Why was the big cat disqualified from the race? Because it was a cheetah.', words_in_joke=14)

In [55]:
def fetch_jokes() -> Joke:
    response = requests.get("https://icanhazdadjoke.com/", headers={"Accept": "application/json"}, timeout=10)
    response.raise_for_status()
    data = response.json()
    return Joke.model_validate(data)

jokes: list[Joke] = []

for i in range(10):
    try:
        fj = fetch_jokes()
        jokes.append(fj)
        print(f"{i+1}/10_ {fj.id}")
    except Exception as e:
        print(f"Failed to fetch {i+1}: {e}")
    if i < 9:
        time.sleep(5)

1/10_ 69xAsrHYDAd
2/10_ GQuzAAXgah
3/10_ FBQK6MexPuc
4/10_ giyXgahV0wc
5/10_ lbU01DljGtc
6/10_ ly5hNR7w5Ed
7/10_ aFtzPRSnbxc
8/10_ 4EBXnjiVKBd
9/10_ HY8xHeiNeFd
10/10_ HY8xHeiNeFd


In [56]:
for j in jokes:
    print(f"{j.joke}")

Why did Mozart kill all his chickens?
Because when he asked them who the best composer was, they'd all say "Bach bach bach!"

Some people say that I never got over my obsession with Phil Collins.
But take a look at me now.
My friend told me that pepper is the best seasoning for a roast, but I took it with a grain of salt.
The great thing about stationery shops is they're always in the same place...
I couldn't get a reservation at the library. They were completely booked.
I tried taking some high resolution photos of local farmland, but they all turned out a bit grainy.
Two fish are in a tank, one turns to the other and says, "how do you drive this thing?"
Why did the sentence fail the driving test? It never came to a full stop.
Leather is great for sneaking around because it's made of hide.
Leather is great for sneaking around because it's made of hide.
